# NB08 — Hessen/Darmstadt Method Ablation (full-tile, no GT)

**Runs on Kaggle T4 GPU.**

Compares 4 methods on two full-tile 5000×5000 images
(`dop20_32_469_5522_1_he`, `dop20_32_476_5524_1_he`), using the **exact same
sliding-window parameters as NB03 v10** (`slide_crop=1024`, `slide_stride=768`,
`prob_thd=0.1`, `confidence_threshold=0.1`) so results are comparable.

| # | Method | Description |
|---|---|---|
| A | ZS-single | Zero-shot, one word per class |
| B | ZS-multi | Zero-shot, our refined multi-synonym per-image prompts (same as NB03) |
| C | ZS-multi + PAMR | B + PAMR boundary refinement (image-guided edge sharpening, 10 iter) |
| D | PTSAM + PAMR | C + Potsdam-trained soft prompts injected into language features |

**No ground truth exists for Darmstadt — this is visual comparison only, no mIoU.**

**Method D caveat:** `soft_prompts.pt` (8 tokens × 256 dims) was trained on
Potsdam's 5cm-GSD imagery with a *different* 5-class vocabulary (impervious,
building, low_veg, tree, car — no runway, no sports courts, no water). Applying
it here to our new Hessen classes at 20cm GSD is an **out-of-domain, unvalidated
experiment** — include it as a visual curiosity, not evidence it helps. If it
looks worse than C, that's expected, not a bug.

**Datasets to attach:** `dummyirl/sam3-weights` · `harish77718/darmstadt-dop20` · `harish77718/ptsam-soft-prompts`

## 1 — Environment setup

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [ ]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX \u2192 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated \u2192 {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned \u2192 {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — 4-method ablation on both full-tile images

Backbone runs once per crop, shared across all 4 methods (same pattern as NB06).
PAMR runs per-crop (not on the full 5000×5000 tile) to avoid OOM.

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, torch, torch.nn.functional as F
import numpy as np
from pathlib import Path
from PIL import Image

sys.stdout.reconfigure(line_buffering=True)

DEVICE               = "cuda"
INPUT_FOLDER         = Path("/kaggle/input/darmstadt-dop20")
OUT_DIR              = Path("/kaggle/working/output"); OUT_DIR.mkdir(parents=True, exist_ok=True)

# Same sliding-window params as NB03 v10, for comparability.
CROP_SIZE            = 1024
STRIDE               = 768
CONFIDENCE_THRESHOLD = 0.1
PROB_THD             = 0.1
BG_IDX               = 255
PAMR_ITER            = 10
PAMR_DILATIONS       = [1, 2, 4]   # per-crop, avoids OOM on full tile

sys.path.insert(0, str(Path('.')))
from pamr import PAMR
pamr_module = PAMR(num_iter=PAMR_ITER, dilations=PAMR_DILATIONS).to(DEVICE).eval()
print(f"PAMR loaded (iter={PAMR_ITER}, dilations={PAMR_DILATIONS}, applied per-crop)", flush=True)

# ── per-image multi-synonym classes (same as NB03 v10) + derived single-word list ──
IMAGE_CLASSES = {
    "dop20_32_469_5522_1_he": {
        "multi": ["building, rooftop",
                   "paved road, street",
                   "airfield runway, landing strip, tarmac",
                   "tree, forest",
                   "farmland, low vegetation, grass",
                   "car, vehicle"],
        "single": ["building", "road", "runway", "tree", "farmland", "car"],
        "colors": np.array([
            [  0,   0, 255],
            [ 80,  80,  80],
            [255,   0,   0],
            [  0, 200,   0],
            [  0, 255, 255],
            [255, 255,   0],
        ], dtype=np.uint8),
    },
    "dop20_32_476_5524_1_he": {
        "multi": ["water body, river, lake",
                   "building, rooftop",
                   "paved road, street",
                   "green sports field, football pitch, grass pitch",
                   "clay sports court, dirt sports ground, tennis court",
                   "tree, low vegetation, grass",
                   "car, vehicle"],
        "single": ["water", "building", "road", "sports field", "clay court", "tree", "car"],
        "colors": np.array([
            [  0, 100, 255],
            [  0,   0, 150],
            [ 80,  80,  80],
            [  0, 200,   0],
            [180, 100,  40],
            [  0, 255, 255],
            [255, 255,   0],
        ], dtype=np.uint8),
    },
}

# ── load soft prompts (method D) ──
candidates = list(Path("/kaggle/input").rglob("soft_prompts.pt"))
if not candidates:
    print("WARNING: soft_prompts.pt not found — method D skipped", flush=True)
    soft_prompts = None
else:
    soft_prompts = torch.load(str(candidates[0]), weights_only=True).to(DEVICE)
    print(f"soft_prompts: {soft_prompts.shape}  (Potsdam-trained, out-of-domain here)", flush=True)

# ── load model ──
from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters(): p.requires_grad = False
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
processor = Sam3Processor(model, confidence_threshold=CONFIDENCE_THRESHOLD, device=DEVICE)

def cache_text(words, with_soft=False):
    cache = []
    with torch.no_grad():
        for word in words:
            te = model.backbone.forward_text([word], device=DEVICE)
            if with_soft and soft_prompts is not None:
                lang_f = te["language_features"]
                soft_m = torch.zeros(1, soft_prompts.shape[0], device=DEVICE,
                                     dtype=te["language_mask"].dtype)
                te["language_features"] = torch.cat([lang_f, soft_prompts.to(dtype=lang_f.dtype)], dim=0)
                te["language_mask"]     = torch.cat([te["language_mask"], soft_m], dim=1)
            cache.append({k: v.cpu() for k, v in te.items()})
    return cache

def collect_class_scores(state, h, w, te_cache, n_classes, device):
    logits = torch.zeros((n_classes, h, w), device=device)
    for cls_idx, te_cpu in enumerate(te_cache):
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items(): state["backbone_out"][k] = v.to(device)
        state["geometric_prompt"] = model._get_dummy_prompt()
        processor._forward_grounding(state)
        scores = torch.zeros((h, w), device=device)
        if state.get("masks_logits") is not None and state["masks_logits"].shape[0] > 0:
            for i in range(state["masks_logits"].shape[0]):
                il = state["masks_logits"][i].squeeze()
                if il.shape != (h, w):
                    il = F.interpolate(il.view(1,1,*il.shape), size=(h,w),
                                       mode="bilinear", align_corners=False).squeeze()
                scores = torch.max(scores, il * state["object_score"][i])
        sem = state["semantic_mask_logits"].squeeze()
        if sem.shape != (h, w):
            sem = F.interpolate(sem.view(1,1,*sem.shape), size=(h,w),
                                mode="bilinear", align_corners=False).squeeze()
        scores = torch.max(scores, sem) * state["presence_score"]
        logits[cls_idx] = torch.max(logits[cls_idx], scores)
    return logits

def pamr_refine_crop(logits_crop, crop_rgb_np):
    img_t = torch.from_numpy(crop_rgb_np.astype(np.float32) / 255.0)\
                 .permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        refined = pamr_module(img_t, logits_crop.unsqueeze(0))
    return refined.squeeze(0)

def make_gaussian_kernel(h, w, dev):
    sy, sx = h/4.0, w/4.0
    y = torch.arange(h, device=dev).float() - (h-1)/2.0
    x = torch.arange(w, device=dev).float() - (w-1)/2.0
    return torch.exp(-y[:,None]**2/(2*sy**2)) * torch.exp(-x[None,:]**2/(2*sx**2))

def run_ablation(source_image, classes):
    img_path = INPUT_FOLDER / f"{source_image}.jpg"
    if not img_path.exists():
        print(f"Not found: {img_path}"); return

    multi_words  = classes["multi"]
    single_words = classes["single"]
    color_map    = classes["colors"]
    n_cls        = len(multi_words)

    print(f"\n=== {source_image}: caching text embeddings ({n_cls} classes) ===", flush=True)
    cache_single = cache_text(single_words, with_soft=False)
    cache_multi  = cache_text(multi_words,  with_soft=False)
    cache_ptsam  = cache_text(multi_words,  with_soft=True) if soft_prompts is not None else None

    img_arr = np.array(Image.open(img_path).convert("RGB"))
    H_full, W_full = img_arr.shape[:2]

    h_grids = max(H_full - CROP_SIZE + STRIDE - 1, 0) // STRIDE + 1
    w_grids = max(W_full - CROP_SIZE + STRIDE - 1, 0) // STRIDE + 1
    total   = h_grids * w_grids
    print(f"Sliding window: {h_grids}x{w_grids}={total} crops", flush=True)

    gauss_k        = make_gaussian_kernel(CROP_SIZE, CROP_SIZE, DEVICE)
    acc_single     = torch.zeros(n_cls, H_full, W_full, device=DEVICE)
    acc_multi      = torch.zeros(n_cls, H_full, W_full, device=DEVICE)
    acc_multi_pamr = torch.zeros(n_cls, H_full, W_full, device=DEVICE)
    acc_ptsam_pamr = torch.zeros(n_cls, H_full, W_full, device=DEVICE) if cache_ptsam else None
    wt_mat         = torch.zeros(H_full, W_full, device=DEVICE)

    for hi in range(h_grids):
        for wi in range(w_grids):
            y1 = hi*STRIDE;  x1 = wi*STRIDE
            y2 = min(y1+CROP_SIZE, H_full);  x2 = min(x1+CROP_SIZE, W_full)
            y1 = max(y2-CROP_SIZE, 0);       x1 = max(x2-CROP_SIZE, 0)

            crop_rgb = img_arr[y1:y2, x1:x2]
            crop_pil = Image.fromarray(crop_rgb)
            h_c, w_c = y2-y1, x2-x1

            with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
                state    = processor.set_image(crop_pil)
                l_single = collect_class_scores(state, h_c, w_c, cache_single, n_cls, DEVICE).float()
                l_multi  = collect_class_scores(state, h_c, w_c, cache_multi,  n_cls, DEVICE).float()
                if cache_ptsam:
                    l_ptsam = collect_class_scores(state, h_c, w_c, cache_ptsam, n_cls, DEVICE).float()

            l_multi_pamr = pamr_refine_crop(l_multi, crop_rgb)
            if cache_ptsam:
                l_ptsam_pamr = pamr_refine_crop(l_ptsam, crop_rgb)

            g = gauss_k[:h_c, :w_c]
            acc_single[:,     y1:y2, x1:x2] += l_single     * g.unsqueeze(0)
            acc_multi[:,      y1:y2, x1:x2] += l_multi      * g.unsqueeze(0)
            acc_multi_pamr[:, y1:y2, x1:x2] += l_multi_pamr * g.unsqueeze(0)
            if cache_ptsam:
                acc_ptsam_pamr[:, y1:y2, x1:x2] += l_ptsam_pamr * g.unsqueeze(0)
            wt_mat[y1:y2, x1:x2] += g

            done = hi*w_grids + wi + 1
            print(f"  {done}/{total} crops", flush=True)

    def finalize(acc):
        p = acc / wt_mat.unsqueeze(0)
        seg = p.argmax(0)
        seg[p.max(0)[0] < PROB_THD] = BG_IDX
        return seg.cpu().numpy()

    seg_A = finalize(acc_single)
    seg_B = finalize(acc_multi)
    seg_C = finalize(acc_multi_pamr)
    seg_D = finalize(acc_ptsam_pamr) if acc_ptsam_pamr is not None else None

    for name, seg in [("A", seg_A), ("B", seg_B), ("C", seg_C), ("D", seg_D)]:
        if seg is not None:
            np.save(str(OUT_DIR / f"{source_image}_method{name}_pred.npy"), seg.astype(np.uint8))

    # ── visualization ──
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    def to_rgb(seg):
        out = np.zeros((*seg.shape, 3), dtype=np.uint8)
        safe = np.where(seg == BG_IDX, 0, seg)
        out[:] = color_map[np.clip(safe, 0, len(color_map)-1)]
        out[seg == BG_IDX] = [30, 30, 30]
        return out

    segs = [seg_A, seg_B, seg_C, seg_D]
    labels = ["A  ZS-single", "B  ZS-multi", "C  ZS-multi+PAMR", "D  PTSAM+PAMR (out-of-domain)"]

    fig, axes = plt.subplots(1, 5, figsize=(30, 7))
    axes[0].imshow(img_arr); axes[0].axis("off"); axes[0].set_title(f"{source_image}.jpg", fontsize=12, fontweight="bold")
    for j, (seg, label) in enumerate(zip(segs, labels), start=1):
        if seg is None:
            axes[j].axis("off"); axes[j].set_title(f"{label}\n(skipped)", fontsize=11)
            continue
        axes[j].imshow(img_arr)
        axes[j].imshow(to_rgb(seg), alpha=0.5)
        axes[j].axis("off")
        axes[j].set_title(label, fontsize=11, fontweight="bold")

    legend_elements = [Patch(facecolor=c/255.0, edgecolor="black", label=n)
                        for n, c in zip(multi_words, color_map)]
    plt.subplots_adjust(bottom=0.2)
    legend = fig.legend(handles=legend_elements, loc="lower center", ncol=min(4, n_cls),
                        fontsize=8, frameon=False, bbox_to_anchor=(0.5, 0.02))

    meta_text = (f"img_size = {W_full}x{H_full}    prob_thd = {PROB_THD}    "
                 f"conf_thd = {CONFIDENCE_THRESHOLD}    slide_stride = {STRIDE}    slide_crop = {CROP_SIZE}")
    footer = fig.text(0.5, 0.09, meta_text, ha="center", fontsize=9, family="monospace")

    out_png = OUT_DIR / f"{source_image}_ablation.png"
    fig.savefig(str(out_png), dpi=150, bbox_inches="tight", bbox_extra_artists=[legend, footer])
    plt.close(fig)
    print(f"Saved: {out_png}", flush=True)

for source_image, classes in IMAGE_CLASSES.items():
    run_ablation(source_image, classes)
PYEOF

### Takeaways — ablation runs

_Fill in after each run: which method looked best per image, and why._

- `dop20_32_469_5522_1_he`:
- `dop20_32_476_5524_1_he`: